# Data Preparation Workflow

### Libraries

In [ ]:
import time
start = time.time()

In [ ]:
# !pip install pandas scikit-learn


In [ ]:
import os, re
import pandas as pd
from sklearn.preprocessing import LabelEncoder

### Chargement des données

- charger les données en DataFrame

In [ ]:
path = input("Entrer le chemin du dataset CSV: ")
    
# vérifier l'existence du dossier ../data et valider le chemin du fichier CSV
default_dir = "../data"
default_path = os.path.join(default_dir, "movies.csv")

os.makedirs(default_dir, exist_ok=True)

# nettoyer la saisie utilisateur
path = (path or "").strip()

# si vide ou extension invalide -> utiliser le fichier par défaut
if not path or not path.lower().endswith(".csv"):
    path = default_path

# si le chemin choisi n'existe pas, tenter le fallback sur le fichier par défaut
if not os.path.exists(path):
    if path != default_path and os.path.exists(default_path):
        print(f"Chemin introuvable. Utilisation du fichier par défaut : {default_path}")
        path = default_path
    else:
        raise FileNotFoundError(f"Le fichier {os.path.basename(default_path)} n'existe pas dans le dossier {default_dir}")

In [ ]:
dataframe = pd.read_csv(path)

In [ ]:
dataframe.info()

In [ ]:
dataframe.columns

In [ ]:
dataframe.head(3)

### Études des données
- choix des features pertinentes pour la modélisation
    + variables pertinentes à la classification
        - durée
        - année de sortie
        - note (rating)
        - nombre de votes
        - revenu brut (gross income)
    + variable cible
        - genre (après uniformisation/encodage)

In [ ]:
sub_dataframe = dataframe[['duration', 'year', 'rating', 'votes', 'gross_income', 'genre']]

- variables pertinentes à la classification

In [ ]:
sub_dataframe[['duration', 'year', 'rating', 'votes', 'gross_income']].head(3)

In [ ]:
sub_dataframe.info()

- variable cible 

In [ ]:
sub_dataframe["genre"].head(10)

### Traitement des données

In [ ]:
duplicates_mask = sub_dataframe.duplicated()
print("Nombre de lignes dupliquées :", duplicates_mask.sum())

In [ ]:
sub_dataframe = sub_dataframe.drop_duplicates()

In [ ]:
sub_dataframe.info()

- valeurs nulles par colonnes numériques

In [ ]:
print("Nombre de valeurs nulles par colonne numérique:")
print(sub_dataframe[['duration', 'year', 'rating', 'votes', 'gross_income']].isnull().sum())

### uniformisation des durées de film

In [ ]:
valeur_uniques_duration = sub_dataframe['duration'].unique()
# valeur_uniques_duration

In [ ]:
def convertir_duree(duree_str):
    if pd.isna(duree_str) or not isinstance(duree_str, str):
        return float('nan')

    duree_str = duree_str.strip().lower().replace(",", "")

    if "h" in duree_str:
        parts = duree_str.split("h", 1)
        heures = int(parts[0].strip()) if parts[0].strip() else 0
        minutes_part = parts[1].replace("min", "").strip() if len(parts) > 1 else ""
        minutes = int(minutes_part) if minutes_part else 0
        return heures * 60 + minutes

    if "min" in duree_str:
        minutes_str = duree_str.replace("min", "").strip()
        return int(minutes_str) if minutes_str else 0

    if duree_str.isdigit():
        return int(duree_str)

    return float('nan')

# test
# for duree in valeur_uniques_duration:
#     print(f"{duree} -> {convertir_duree(duree)}")

In [ ]:
sub_dataframe.loc[:, 'duration'] = sub_dataframe['duration'].apply(convertir_duree)

In [ ]:
# compter le nombre de films par genre ayant une durée de 0 minute
films_duree_zero = sub_dataframe[sub_dataframe['duration'] == 0]
print("Nombre de films avec durée de 0 minute par genre :")
print(films_duree_zero['genre'].value_counts())

In [ ]:
sub_dataframe.loc[:, 'duration'] = sub_dataframe['duration'].apply(lambda x: x if x > 0 else float('nan')) # remplacer les 0 par NaN

In [ ]:
sub_dataframe.head(10)

### uniformisation des années de sortie

In [ ]:
def convertir_annee(annee_str):
    if pd.isna(annee_str):
        return float('nan')

    # gérer aussi les cas numériques déjà propres
    if isinstance(annee_str, (int, float)) and not pd.isna(annee_str):
        annee_int = int(annee_str)
        return annee_int if 1800 <= annee_int <= 2100 else float('nan')

    if not isinstance(annee_str, str):
        return float('nan')

    texte = annee_str.strip()

    # récupérer la première année à 4 chiffres trouvée dans la chaîne
    match = re.search(r'(?<!\d)(\d{4})(?!\d)', texte)
    if match:
        annee_int = int(match.group(1))
        return annee_int if 1800 <= annee_int <= 2100 else float('nan')

    return float('nan')

# test (aperçu limité pour éviter un affichage trop long)
# valeur_uniques_year = sub_dataframe['year'].dropna().unique()
# for annee in valeur_uniques_year[:50]:
#     print(f"{annee} -> {convertir_annee(annee)}")

# print("Valeurs 'year' non converties (NaN) :", sub_dataframe['year'].isna().sum())

In [ ]:
sub_dataframe.loc[:, 'year'] = sub_dataframe['year'].apply(convertir_annee)

In [ ]:
sub_dataframe.head(10)

### uniformisation des votes

In [ ]:
sub_dataframe["votes"].unique()

In [ ]:
def convertir_votes(votes_str):
    if pd.isna(votes_str):
        return float('nan')

    # cas numérique direct (int/float)
    if isinstance(votes_str, (int, float)):
        votes_float = float(votes_str)
        if votes_float < 0 or not votes_float.is_integer():
            return float('nan')
        return int(votes_float)

    # cas non string non numérique
    if not isinstance(votes_str, str):
        return float('nan')

    # nettoyage texte
    texte = votes_str.strip().replace(",", "")
    if not texte:
        return float('nan')

    # conversion robuste (gère "124.0", "22792", etc.)
    try:
        votes_float = float(texte)
    except ValueError:
        return float('nan')

    if votes_float < 0 or not votes_float.is_integer():
        return float('nan')

    return int(votes_float)

# test
# valeur_uniques_votes = sub_dataframe['votes'].dropna().unique()
# for votes in valeur_uniques_votes[:50]:
#     print(f"{votes} -> {convertir_votes(votes)}")

In [ ]:
sub_dataframe.loc[:, 'votes'] = sub_dataframe['votes'].apply(convertir_votes)

### uniformisation des revenus bruts

In [ ]:
def convertir_gross_income(gross_str):
    if pd.isna(gross_str):
        return float('nan')

    # déjà numérique
    if isinstance(gross_str, (int, float)):
        return float(gross_str)

    if not isinstance(gross_str, str):
        return float('nan')

    s = gross_str.strip()
    if not s:
        return float('nan')

    # enlever séparateurs de milliers et caractères non numériques
    s = s.replace(",", "")
    s = re.sub(r"[^\d.\-]", "", s)

    if s in {"", ".", "-", "-."}:
        return float('nan')

    try:
        return float(s)
    
    except ValueError:
        return float('nan')

In [ ]:
sub_dataframe["gross_income"].unique()

In [ ]:
sub_dataframe.loc[:, 'gross_income'] = sub_dataframe['gross_income'].apply(convertir_gross_income)

In [ ]:
# combien de vide/NaN dans cette colonne ?
# print(sub_dataframe['gross_income'].isnull().sum())

In [ ]:
sub_dataframe.loc[:, 'gross_income'] = sub_dataframe['gross_income'].apply(lambda x: x if x >= 0 else float('nan')) # remplacer les 0 par NaN

In [ ]:
# combien de vide/NaN dans cette colonne ?
# print(sub_dataframe['gross_income'].isnull().sum())

In [ ]:
sub_dataframe.head(10)

### uniformisation des genres

In [ ]:
print("Genres avant uniformisation :",
      sub_dataframe['genre'].unique(),
      "\nNombre de genres uniques avant uniformisation :",
      len(sub_dataframe['genre'].unique()))

In [ ]:
def extract_first_genre(x):
    if pd.isna(x) or not isinstance(x, str):
        return float('nan')
    return x.split(',')[0].strip()

# test
# for genre in sub_dataframe['genre'].dropna().unique()[:50]:
#     print(f"{genre} -> {extract_first_genre(genre)}")

- filtrer les genres **multiples** selon une liste de genres indésirables
    1. prendre en entrée une chaîne de caractères représentant les genres d’un film, séparés par des virgules.
    2. conserver uniquement les genres qui ne figurent pas dans la liste des genres à supprimer (passée en paramètres).

In [ ]:
# fonction : tu prends en entrée une chaîne de caractères représentant les genres d'un film séparés par des virgules et tu vas garder uniquement les genres pas présents dans la liste en parametres
def filter_genres(genre_str, genres_to_remove):
    if pd.isna(genre_str) or not isinstance(genre_str, str):
        return float('nan')

    genres = [g.strip() for g in genre_str.split(",")]
    filtered_genres = [g for g in genres if g not in genres_to_remove]

    return ", ".join(filtered_genres) if filtered_genres else float('nan')

# test
# genres_to_remove = ['Reality-TV', 'Talk-Show', 'Game-Show', 'News', 'Film-Noir', 'Adult', 'Short']
# for genre_str in sub_dataframe['genre'].unique():
#     print(f"{genre_str} -> {filter_genres(genre_str, genres_to_remove)}")

In [ ]:
liste_genres_a_retirer = {
    "Animation",
    "Reality-TV",
    "Talk-Show",
    "Game-Show",
    "Documentary",
    "News",
    "Film-Noir",
    "Adult",
    "Short",
    "Musical",
    "War",
    # "Romance",
    # "Crime",
    # "Sci-Fi",
    # "Family",
    # "Fantasy",
    # "Mystery",
    # "Drama",
}

sub_dataframe.loc[:, 'genre'] = sub_dataframe['genre'].apply(
    filter_genres,
    genres_to_remove=liste_genres_a_retirer
)

In [ ]:
sub_dataframe.head(10)

In [ ]:
sub_dataframe.value_counts('genre')

In [ ]:
# supprimer les lignes ayant encore des genres multiples restants car déjà assez de lignes dans les genres uniques
sub_dataframe.loc[:, 'genre'] = sub_dataframe['genre'].apply(lambda x: x if isinstance(x, str) and ',' not in x else float('nan'))

In [ ]:
sub_dataframe.value_counts('genre')

- garder uniquement le premier genre d’un film (après filtrage des genres indésirables)

In [ ]:
# sub_dataframe.loc[:, 'genre'] = sub_dataframe['genre'].apply(extract_first_genre)

In [ ]:
# sub_dataframe.head(10)

In [ ]:
# comparer la suppresion actuelle
# et l'option 50% des features avec du vide → passer par la moyenne

In [ ]:
# compter les cases vides/Nan selon les colonnes
sub_dataframe.isnull().sum()

In [ ]:
numerical_cols = ['duration', 'year', 'rating', 'votes', 'gross_income']

for col in numerical_cols:
    sub_dataframe.loc[:, col] = pd.to_numeric(sub_dataframe[col], errors='coerce').astype('float64')

In [ ]:
sub_dataframe.loc[:, "rating"] = pd.to_numeric(sub_dataframe.loc[:, "rating"], errors='coerce').astype(float)
sub_dataframe.loc[:, "votes"] = pd.to_numeric(sub_dataframe.loc[:, "votes"], errors='coerce').astype(float)
sub_dataframe.loc[:, "gross_income"] = pd.to_numeric(sub_dataframe.loc[:, "gross_income"], errors='coerce').astype(float)
sub_dataframe.loc[:, "duration"] = pd.to_numeric(sub_dataframe.loc[:, "duration"], errors='coerce').astype(float)
sub_dataframe.loc[:, "year"] = pd.to_numeric(sub_dataframe.loc[:, "year"], errors='coerce').astype(float)

In [ ]:
# convertir proprement en numérique pour éviter le warning de downcasting
sub_dataframe.loc[:, numerical_cols] = sub_dataframe[numerical_cols].apply(pd.to_numeric, errors='coerce')

In [ ]:
# # moyenne par colonne puis arrondi (même logique que ton code initial)
# mean_values = sub_dataframe[numerical_cols].mean().round()

# # remplissage des NaN sans chained assignment
# sub_dataframe.loc[:, numerical_cols] = sub_dataframe[numerical_cols].fillna(mean_values)

In [ ]:
sub_dataframe.value_counts('genre')

### suppression des lignes avec des valeurs manquantes

In [ ]:
output = sub_dataframe.copy()
output.dropna(inplace=True)
# output.head(10)
# output.info()
output.value_counts('genre')

- transformer les genres minoritaires en "Other" (optionnel, à discuter) selon un seuil de fréquence défini

In [ ]:
# si le nombre lignes par genre est supérieur à 1000,
# → garder le genre
# → sinon le remplacer par "Other"
# SEUIL_MINIMUM = 1000
# genre_counts = output['genre'].value_counts()
# genres_to_keep = genre_counts[genre_counts >= SEUIL_MINIMUM].index
# output.loc[~output['genre'].isin(genres_to_keep), 'genre'] = "Other"

# output.value_counts('genre')

In [ ]:
sub_dataframe = output.copy()
sub_dataframe.value_counts('genre')

- convertir les genres (**valeur prédite**) en format numérique

In [ ]:
label_encoder = LabelEncoder()

In [ ]:
# Encodage des genres non nuls

valid_genres = sub_dataframe["genre"].dropna().astype(str)

label_encoder.fit(valid_genres)

genre_map = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

sub_dataframe["genre_encoded"] = sub_dataframe["genre"].map(genre_map)
sub_dataframe["genre_encoded"] = sub_dataframe["genre_encoded"].astype("Int64")

In [ ]:
sub_dataframe.dtypes

In [ ]:
print("Nombre de lignes par genre encodé :")
print(sub_dataframe[['genre', 'genre_encoded']].value_counts())

In [ ]:
# choisir aléatoirement 5000 lignes par genre pour éviter les déséquilibres extrêmes
def sample_by_genre(df, genre_col, n_samples):
    return df.groupby(genre_col).apply(lambda x: x.sample(n=min(len(x), n_samples), random_state=42)).reset_index(drop=True)

# test
# sampled_df = sample_by_genre(sub_dataframe, 'genre_encoded', 5000)
# print(sampled_df['genre_encoded'].value_counts())

In [ ]:
# sub_dataframe = sample_by_genre(sub_dataframe, 'genre_encoded', 5000)
# sub_dataframe['genre_encoded'].value_counts()

In [ ]:
sub_dataframe["rating"] = sub_dataframe["rating"].astype(float)
sub_dataframe["votes"] = sub_dataframe["votes"].astype(float)
sub_dataframe["gross_income"] = sub_dataframe["gross_income"].astype(float)
sub_dataframe["duration"] = sub_dataframe["duration"].astype(float)
sub_dataframe["year"] = sub_dataframe["year"].astype(float)

sub_dataframe["genre"] = sub_dataframe["genre"].astype("category")

In [ ]:
sub_dataframe.info()

In [ ]:
sub_dataframe["genre"].value_counts()

### corrélation entre les variables cibles

In [ ]:
correlation_data = sub_dataframe.drop(columns=['genre', 'genre_encoded']).corr()
correlation_data

### Exporter le DataFrame en CSV

In [ ]:
sub_dataframe.drop(columns=['genre']).to_csv('../data/movies_cleaned.csv', index=False)
end = time.time() - start

print(f"Temps d'exécution : {end:.2f} secondes")